In [ ]:
!pip install langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.9 MB/s eta 0:00:00


In [ ]:
"""
RAG-Based Organization Chatbot System with Turkish Gemma Model
===============================================================
FIXED VERSION: Updated for LangChain v1 (2025) with correct imports
"""

# ============================================================================
# STEP 1: INSTALL REQUIRED LIBRARIES (FIXED FOR v1)
# ============================================================================
try:
  import os
  import json
  import re
  from typing import List, Dict
  from pathlib import Path

  from langchain_core.documents import Document
  from langchain_chroma import Chroma
  from langchain_huggingface import HuggingFaceEmbeddings
  from langchain_core.prompts import PromptTemplate
  from langchain_core.language_models.llms import LLM
  from langchain_classic.chains import RetrievalQA

  from llama_cpp import Llama
except:
  # Ensure torch is installed
  !pip install -q torch

  # Install llama-cpp-python with GPU support
  print("⏳ Installing llama-cpp-python with GPU support...")
  !pip uninstall -y llama-cpp-python 2>/dev/null
  !pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

  # CRITICAL FIX: Install correct LangChain v1 packages
  print("⏳ Installing LangChain v1 packages...")
  !pip uninstall -y langchain langchain-community langchain-chroma 2>/dev/null

  # Core packages for v1
  !pip install -q langchain>=1.0.0
  !pip install -q langchain-core>=1.0.0
  !pip install -q langchain-chroma>=0.2.0
  !pip install -q langchain-huggingface>=0.1.0

  # For legacy RetrievalQA (moved to langchain-classic in v1)
  !pip install -q langchain-classic

  # Other dependencies
  !pip install -q chromadb sentence-transformers huggingface-hub

  print("✅ All libraries installed successfully!")

  # Debug: Verify installation
  import torch
  if torch.cuda.is_available():
      print(f"✅ PyTorch: Found GPU! ({torch.cuda.get_device_name(0)})")
  else:
      print("❌ PyTorch: No GPU found")

  try:
      import llama_cpp
      print(f"✅ llama-cpp-python: v{llama_cpp.__version__}")
  except ImportError as e:
      print(f"❌ llama-cpp-python import failed: {e}")

# ============================================================================
# STEP 2: IMPORT LIBRARIES (FIXED FOR v1)
# ============================================================================

import os
import json
import re
from typing import List, Dict
from pathlib import Path

# ✅ FIXED: Correct v1 imports
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.language_models.llms import LLM

# ✅ CRITICAL FIX: RetrievalQA moved to langchain-classic in v1
from langchain_classic.chains import RetrievalQA

# Llama CPP
from llama_cpp import Llama

print("✅ Libraries imported successfully with v1 paths!")

# ============================================================================
# STEP 3: CONFIGURATION
# ============================================================================

class Config:
    """Configuration for the model"""

    # Embedding model (Turkish support)
    EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

    # Vector database
    VECTOR_DB_PATH = "./chroma_db"
    COLLECTION_NAME = "org_knowledge_turkish"

    # Turkish Gemma Model Configuration
    GEMMA_REPO_ID = "ytu-ce-cosmos/Turkish-Gemma-9b-T1-GGUF"
    GEMMA_FILENAME = "*Q4_K.gguf"

    # Gemma inference parameters
    GEMMA_PARAMS = {
        "n_gpu_layers": -1,
        "n_threads": 4,
        "n_ctx": 8192,
        "n_predict": 2048,
        "top_k": 40,
        "top_p": 0.90,
        "temp": 0.1,
        "repeat_penalty": 1.1,
    }

    # Retrieval settings
    TOP_K_RESULTS = 10

config = Config()

# ============================================================================
# STEP 4: DATA LOADING (Supports both old and new formats)
# ============================================================================

class QADataLoader:
    """Loads pre-chunked Q&A data from various formats"""

    def __init__(self):
        self.stats = {
            "total_qa_pairs": 0,
            "by_category": {},
            "by_file": {},
            "by_format": {"old_format": 0, "new_format": 0}
        }

    def clean_text(self, text: str) -> str:
        """Helper to clean text before saving to Document"""
        if not text:
            return ""
        text = re.sub(r'\[cite:.*?\]', '', text)
        text = re.sub(r'\\', '', text)
        return text.strip()

    def detect_json_format(self, data: dict) -> str:
        """Detect if JSON is old format or new format"""
        if 'data' in data and isinstance(data['data'], list):
            return "old_format"
        elif isinstance(data, list):
            return "new_format_list"
        elif isinstance(data, dict) and any(key in data for key in ['question', 'soru', 'answer', 'cevap']):
            return "new_format_single"
        else:
            return "unknown"

    def load_json_qa(self, file_path: str) -> List[Document]:
        """Load Q&A pairs from JSON - HANDLES BOTH OLD AND NEW FORMATS"""
        documents = []

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            format_type = self.detect_json_format(data)

            path_parts = Path(file_path).parts
            main_category = "Genel"
            sub_category = Path(file_path).stem

            for part in path_parts:
                if part.startswith(('01_', '02_', '03_', '04_', '05_')):
                    main_category = part.split('_', 1)[1] if '_' in part else part
                    break

            qa_pairs = []

            if format_type == "old_format":
                qa_pairs = data.get('data', [])
                self.stats["by_format"]["old_format"] += len(qa_pairs)
                print(f"  📋 Old format detected in {Path(file_path).name}")

            elif format_type == "new_format_list":
                qa_pairs = data
                self.stats["by_format"]["new_format"] += len(qa_pairs)
                print(f"  📋 New format (list) detected in {Path(file_path).name}")

            elif format_type == "new_format_single":
                qa_pairs = [data]
                self.stats["by_format"]["new_format"] += 1
                print(f"  📋 New format (single) detected in {Path(file_path).name}")

            else:
                print(f"  ⚠️ Unknown format in {Path(file_path).name}, skipping...")
                return []

            for item in qa_pairs:
                question = item.get('question') or item.get('soru', '')
                answer = item.get('answer') or item.get('cevap', '')

                if not question or not answer:
                    continue

                clean_q = self.clean_text(str(question))
                clean_a = self.clean_text(str(answer))

                content = f"Soru: {clean_q}\n\nCevap: {clean_a}"

                keywords = item.get('keywords') or item.get('anahtar_kelimeler', [])
                if isinstance(keywords, list):
                    keywords_str = ", ".join(str(k) for k in keywords)
                else:
                    keywords_str = str(keywords)

                related_questions = item.get('related_questions') or item.get('iliskili_sorular', [])
                if isinstance(related_questions, list):
                    related_str = " | ".join(str(q) for q in related_questions)
                else:
                    related_str = str(related_questions)

                item_category = item.get('category') or item.get('kategori', main_category)
                priority = item.get('priority') or item.get('oncelik', 'medium')

                metadata = {
                    "question": str(question),
                    "answer": str(answer),
                    "main_category": str(main_category),
                    "sub_category": str(sub_category),
                    "category": str(item_category),
                    "keywords": keywords_str,
                    "related_questions": related_str,
                    "priority": str(priority),
                    "source": str(file_path),
                    "file_name": str(Path(file_path).name),
                    "format_type": format_type
                }

                documents.append(Document(page_content=content, metadata=metadata))

            self.stats["by_file"][Path(file_path).name] = len(documents)
            if main_category not in self.stats["by_category"]:
                self.stats["by_category"][main_category] = 0
            self.stats["by_category"][main_category] += len(documents)

            print(f"✅ Loaded {len(documents)} Q&A pairs from {Path(file_path).name}")
            return documents

        except Exception as e:
            print(f"❌ Error loading {file_path}: {str(e)}")
            import traceback
            traceback.print_exc()
            return []

    def load_from_directory(self, dir_path: str) -> List[Document]:
        """Load all Q&A files from a directory"""
        all_docs = []

        for file_path in sorted(Path(dir_path).rglob('*.json')):
            docs = self.load_json_qa(str(file_path))
            all_docs.extend(docs)

        self.stats["total_qa_pairs"] = len(all_docs)

        print(f"\n{'='*70}")
        print(f"✅ TOPLAM {len(all_docs)} Q&A ÇİFTİ YÜKLENDİ")
        print(f"{'='*70}")

        return all_docs

    def load_from_multiple_directories(self, dir_paths: List[str]) -> List[Document]:
        """Load Q&A files from MULTIPLE directories and combine them"""
        all_docs = []

        print(f"\n{'='*70}")
        print(f"📂 LOADING FROM {len(dir_paths)} DIRECTORIES")
        print(f"{'='*70}\n")

        for dir_path in dir_paths:
            if not Path(dir_path).exists():
                print(f"⚠️ Directory not found: {dir_path}, skipping...")
                continue

            print(f"\n📁 Processing directory: {dir_path}")
            docs = self.load_from_directory(dir_path)
            all_docs.extend(docs)
            print(f"  ✅ Added {len(docs)} Q&A pairs from {dir_path}")

        self.stats["total_qa_pairs"] = len(all_docs)

        return all_docs

    def print_statistics(self):
        """Print detailed loading statistics"""
        print("\n" + "="*70)
        print("📊 VERİ YÜKLEME İSTATİSTİKLERİ")
        print("="*70)

        print(f"\n🔢 Toplam Q&A Çifti: {self.stats['total_qa_pairs']}")

        print(f"\n📋 Format Dağılımı:")
        for format_name, count in self.stats['by_format'].items():
            if count > 0:
                percentage = (count / self.stats['total_qa_pairs'] * 100) if self.stats['total_qa_pairs'] > 0 else 0
                print(f"  • {format_name}: {count} çift (%{percentage:.1f})")

        print("\n📁 Kategorilere Göre Dağılım:")
        for category, count in sorted(self.stats['by_category'].items()):
            percentage = (count / self.stats['total_qa_pairs'] * 100) if self.stats['total_qa_pairs'] > 0 else 0
            print(f"  • {category}: {count} çift (%{percentage:.1f})")

        print("="*70 + "\n")

# ============================================================================
# STEP 5: VECTOR STORE
# ============================================================================

class VectorStore:
    """Manages the vector database"""

    def __init__(self, embedding_model_name: str, db_path: str, collection_name: str):
        print("⏳ Initializing Turkish embedding model...")

        import torch
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"  (Using device: {device} for embeddings)")

        self.embeddings = HuggingFaceEmbeddings(
            model_name=embedding_model_name,
            model_kwargs={'device': device}
        )
        self.db_path = db_path
        self.collection_name = collection_name
        self.vectorstore = None
        print("✅ Embedding model ready!")

    def create_from_documents(self, documents: List[Document]):
        """Create a new vector store from Q&A documents"""
        print(f"⏳ Creating vector embeddings for {len(documents)} Q&A pairs...")

        import os
        import shutil

        if os.path.exists(self.db_path):
            try:
                shutil.rmtree(self.db_path)
                print(f"  Removed old database at {self.db_path}")
            except Exception as e:
                print(f"  Warning: Could not remove old database: {e}")

        os.makedirs(self.db_path, exist_ok=True)

        import chromadb
        from chromadb.config import Settings

        chroma_client = chromadb.PersistentClient(
            path=self.db_path,
            settings=Settings(
                anonymized_telemetry=False,
                allow_reset=True
            )
        )

        self.vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            client=chroma_client,
            collection_name=self.collection_name
        )

        print("✅ Vector store created and persisted!")
        return self.vectorstore

# ============================================================================
# STEP 6: TURKISH GEMMA LLM WRAPPER
# ============================================================================

class TurkishGemmaLLM(LLM):
    """Custom LLM wrapper for Turkish Gemma model"""

    model: Llama = None

    def __init__(self, repo_id: str, filename: str, **kwargs):
        super().__init__()
        print("⏳ Downloading and loading Turkish Gemma model...")
        print("   (This may take several minutes on first run)")

        self.model = Llama.from_pretrained(
            repo_id=repo_id,
            filename=filename,
            verbose=False,
            **config.GEMMA_PARAMS
        )

        print("✅ Turkish Gemma model loaded!")

        if config.GEMMA_PARAMS.get('n_gpu_layers', 0) > 0 or config.GEMMA_PARAMS.get('n_gpu_layers') == -1:
            print("   🚀 Model is using GPU acceleration!")

    @property
    def _llm_type(self) -> str:
        return "turkish_gemma"

    def _call(self, prompt: str, stop=None) -> str:
        """Generate and clean response"""

        formatted_prompt = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"

        response = self.model(
            formatted_prompt,
            stop=["<end_of_turn>", "</s>"],
            max_tokens=config.GEMMA_PARAMS['n_predict'],
            temperature=config.GEMMA_PARAMS['temp'],
            top_p=config.GEMMA_PARAMS['top_p'],
            top_k=config.GEMMA_PARAMS['top_k'],
            repeat_penalty=config.GEMMA_PARAMS['repeat_penalty']
        )

        raw_text = response['choices'][0]['text']

        cleaned_text = re.sub(r'<think>.*?(?:</think>|$)', '', raw_text, flags=re.DOTALL)
        cleaned_text = re.sub(r'\[cite.*?\]', '', cleaned_text, flags=re.DOTALL)
        cleaned_text = re.sub(r'\\', '', cleaned_text)

        disclaimer_patterns = [
            r'\*\(Not:.*?\)',
            r'\(Not:.*?\)',
            r'\* Bilgi tabanımızda.*',
            r'Verilen bilgiler arasında.*',
            r'Bağlamda.*',
        ]

        for pattern in disclaimer_patterns:
            cleaned_text = re.sub(pattern, '', cleaned_text, flags=re.IGNORECASE | re.MULTILINE)

        cleaned_text = cleaned_text.strip()

        if not cleaned_text:
            return "Üzgünüm, bir teknik aksaklık nedeniyle cevabı oluşturamadım. Lütfen tekrar deneyin."

        return cleaned_text

# ============================================================================
# STEP 7: RAG PIPELINE
# ============================================================================

class TurkishRAGChatbot:
    """RAG chatbot using Turkish Gemma model"""

    def __init__(self, vectorstore):
        """Initialize the RAG chatbot with Turkish Gemma"""

        self.llm = TurkishGemmaLLM(
            repo_id=config.GEMMA_REPO_ID,
            filename=config.GEMMA_FILENAME
        )

        self.prompt_template = """
<bos><start_of_turn>user
Sen İpekyolu Girişimci Kuluçka Merkezi'nin resmi yapay zeka asistanısın.
Adın: İpekGPT.

GÖREVİN:
Sana verilen bilgileri (Aşağıdaki VERİLER kısmını) **kendi bilginmiş gibi** kabul et ve kullanıcıya doğrudan cevap ver.

KURALLAR:
1. "Bağlamdaki bilgilere göre", "Verilere göre", "Bilgi tabanına göre", "Metinde yazdığı gibi" gibi ifadeler KESİNLİKLE KULLANMA.
2. Doğrudan cevabı ver.
3. Listeleri madde işaretleri ile düzenle.
4. Bilgi VERİLER kısmında yoksa, "Bu konuda şu an güncel bilgim bulunmuyor" de.

VERİLER:
{context}

SORU: {question}<end_of_turn>
<start_of_turn>model
"""

        self.PROMPT = PromptTemplate(
            template=self.prompt_template,
            input_variables=["context", "question"]
        )

        self.qa_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=vectorstore.as_retriever(
                search_type="mmr",
                search_kwargs={
                    'k': config.TOP_K_RESULTS,
                    'fetch_k': 10
                }
            ),
            return_source_documents=True,
            chain_type_kwargs={"prompt": self.PROMPT}
        )

        print("✅ Turkish RAG Chatbot initialized and ready!")

    def ask(self, question: str, show_sources: bool = True) -> Dict:
        """Ask a question and get an answer"""
        print(f"\n{'='*70}")
        print(f"❓ Soru: {question}")
        print(f"{'='*70}")

        try:
            result = self.qa_chain.invoke({"query": question})

            answer = result['result']
            sources = result['source_documents']

            print(f"\n💡 Cevap:\n{answer}")
            print(f"\n{'='*70}\n")

            if show_sources and sources:
                print(f"\n📚 Bilgi tabanından {len(sources)} kaynak kullanıldı:\n")
                for i, doc in enumerate(sources, 1):
                    main_cat = doc.metadata.get('main_category', 'N/A')
                    sub_cat = doc.metadata.get('sub_category', 'N/A')
                    priority = doc.metadata.get('priority', 'N/A')
                    print(f"  Kaynak {i}: [{main_cat} > {sub_cat}] (Öncelik: {priority})")

            return {
                "answer": answer,
                "sources": sources,
                "num_sources": len(sources),
                "categories_used": [doc.metadata.get('main_category') for doc in sources]
            }

        except Exception as e:
            print(f"\n❌ Hata oluştu: {str(e)}")
            import traceback
            traceback.print_exc()
            return {
                "answer": "Bir hata oluştu. Lütfen tekrar deneyin.",
                "sources": [],
                "num_sources": 0,
                "categories_used": []
            }

# ============================================================================
# STEP 8: MAIN WORKFLOW
# ============================================================================

def build_turkish_rag_system_dual_format(old_data_path: str = None, new_data_path: str = None):
    """Build the Turkish RAG system with BOTH old and new format data"""

    print("=" * 70)
    print("🚀 İPEKYOLU RAG CHATBOT SİSTEMİ KURULUMU (DUAL FORMAT)")
    print("=" * 70)

    print("\n📂 ADIM 1: Her iki format için Q&A verisi yükleniyor...")

    loader = QADataLoader()

    paths_to_load = []
    if old_data_path and Path(old_data_path).exists():
        paths_to_load.append(old_data_path)
    if new_data_path and Path(new_data_path).exists():
        paths_to_load.append(new_data_path)

    if not paths_to_load:
        print("❌ Hiçbir veri dizini bulunamadı!")
        return None

    documents = loader.load_from_multiple_directories(paths_to_load)

    if not documents:
        print("❌ Hiç Q&A verisi yüklenemedi!")
        return None

    loader.print_statistics()

    print("\n🧠 ADIM 2: Vektör veritabanı oluşturuluyor...")
    vector_store = VectorStore(
        embedding_model_name=config.EMBEDDING_MODEL,
        db_path=config.VECTOR_DB_PATH,
        collection_name=config.COLLECTION_NAME
    )
    vectorstore = vector_store.create_from_documents(documents)

    print("\n🤖 ADIM 3: Turkish Gemma RAG chatbot başlatılıyor...")
    chatbot = TurkishRAGChatbot(vectorstore)

    print("\n" + "=" * 70)
    print("✅ İPEKYOLU RAG SİSTEMİ HAZIR! (Both formats loaded)")
    print("=" * 70)

    return chatbot

def batch_test_questions(chatbot_instance, questions_list):
    """Tests the chatbot with a batch of questions and prints the results."""
    print("\n" + "=" * 70)
    print("🧪 BATCH TEST QUESTIONS")
    print("=" * 70)

    results = []
    for i, question in enumerate(questions_list):
        print(f"\n--- Soru {i+1}/{len(questions_list)} ---")
        response = chatbot_instance.ask(question, show_sources=False)
        results.append({
            "question": question,
            "answer": response.get('answer'),
            "num_sources": response.get('num_sources'),
            "categories_used": response.get('categories_used')
        })

    print("\n" + "=" * 70)
    print("📊 BATCH TEST SONUÇLARI")
    print("=" * 70)
    for res in results:
        print(f"\nSoru: {res['question']}")
        print(f"Cevap: {res['answer'][:200]}...")
        print(f"Kullanılan kaynak sayısı: {res['num_sources']}")
        print(f"Kullanılan kategoriler: {', '.join(set(res['categories_used'])) if res['categories_used'] else 'Yok'}")
        print("-" * 30)

    return results

# ============================================================================
# STEP 9: USAGE EXAMPLE WITH PATH VERIFICATION
# ============================================================================

import os

# CRITICAL: Verify paths before running
def verify_and_get_paths():
    """Verify data paths exist and return correct paths"""

    # Possible path variations
    possible_paths = [
        ('./ipekgpt/data', './ipekgpt/IPEKYOLU_RAG_VERISETI'),
        ('ipekgpt/data', 'ipekgpt/IPEKYOLU_RAG_VERISETI'),
        ('/content/ipekgpt/data', '/content/ipekgpt/IPEKYOLU_RAG_VERISETI'),
        ('./data', './IPEKYOLU_RAG_VERISETI'),
    ]

    print("🔍 Verifying data paths...")
    print(f"Current working directory: {os.getcwd()}")
    print(f"\nDirectory contents:")
    for item in os.listdir('.'):
        print(f"  - {item}")

    # Check if ipekgpt folder exists
    if os.path.exists('ipekgpt'):
        print(f"\n✅ Found 'ipekgpt' folder")
        print(f"Contents of ipekgpt/:")
        for item in os.listdir('ipekgpt'):
            print(f"  - {item}")

    # Try each path combination
    for old_path, new_path in possible_paths:
        old_exists = os.path.exists(old_path)
        new_exists = os.path.exists(new_path)

        print(f"\nTrying paths:")
        print(f"  Old format: {old_path} - {'✅ EXISTS' if old_exists else '❌ NOT FOUND'}")
        print(f"  New format: {new_path} - {'✅ EXISTS' if new_exists else '❌ NOT FOUND'}")

        if old_exists or new_exists:
            return (old_path if old_exists else None,
                    new_path if new_exists else None)

    # If no paths found, list what we have
    print("\n❌ Could not find data directories!")
    print("Please check your folder structure.")
    return None, None

# Get verified paths
old_data_path, new_data_path = verify_and_get_paths()

if old_data_path or new_data_path:
    print(f"\n{'='*70}")
    print("✅ Found data directories:")
    if old_data_path:
        print(f"  Old format: {old_data_path}")
    if new_data_path:
        print(f"  New format: {new_data_path}")
    print(f"{'='*70}\n")

    # Build the system
    chatbot = build_turkish_rag_system_dual_format(
        old_data_path=old_data_path,
        new_data_path=new_data_path
    )

    if chatbot:
        print("\n" + "=" * 70)
        print("📖 İPEKYOLU RAG MODÜLÜ YÜKLENDİ - Kullanıma hazır!")
        print("=" * 70)

        # Test the system
        print("\n🧪 Testing with a sample question...")
        chatbot.ask("Kuluçka merkezi nedir?")
else:
    print("\n❌ ERROR: Could not find any data directories!")
    print("Please ensure your data folders are in the correct location.")

⏳ Installing llama-cpp-python with GPU support...
Found existing installation: llama_cpp_python 0.3.16
Uninstalling llama_cpp_python-0.3.16:
  Successfully uninstalled llama_cpp_python-0.3.16
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 551.3/551.3 MB 1.2 MB/s eta 0:00:00
⏳ Installing LangChain v1 packages...
Found existing installation: langchain 1.1.2
Uninstalling langchain-1.1.2:
  Successfully uninstalled langchain-1.1.2
Found existing installation: langchain-chroma 1.0.0
Uninstalling langchain-chroma-1.0.0:
  Successfully uninstalled langchain-chroma-1.0.0
✅ All libraries installed successfully!
✅ PyTorch: Found GPU! (Tesla T4)
✅ llama-cpp-python: v0.3.16
✅ Libraries imported successfully with v1 paths!
🔍 Verifying data paths...
Current working directory: /content/ipekgpt

Directory contents:
  - ipekyolugenclik_temiz_veri.csv
  - ipekyolu_rag_generator.py
  - .git
  - haberler.json
  - README.md
  - =1.0.0
  - raw data
  - =0.1.0
  - İpelYoluGPT.ipynb
  - =0.2.0
  - ipekyolugencl

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model ready!
⏳ Creating vector embeddings for 405 Q&A pairs...
✅ Vector store created and persisted!

🤖 ADIM 3: Turkish Gemma RAG chatbot başlatılıyor...
⏳ Downloading and loading Turkish Gemma model...
   (This may take several minutes on first run)


./Turkish-Gemma-9b-T1.Q4_K.gguf:   0%|          | 0.00/5.76G [00:00<?, ?B/s]

llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


✅ Turkish Gemma model loaded!
   🚀 Model is using GPU acceleration!
✅ Turkish RAG Chatbot initialized and ready!

✅ İPEKYOLU RAG SİSTEMİ HAZIR! (Both formats loaded)

📖 İPEKYOLU RAG MODÜLÜ YÜKLENDİ - Kullanıma hazır!

🧪 Testing with a sample question...

❓ Soru: Kuluçka merkezi nedir?

💡 Cevap:
Kuluçka merkezi, **genç girişimcilere ve yenilikçi fikirlere destek sağlayan bir eğitim ve iş geliştirme merkezidir**. İşlevi şu şekildedir:

- **Fikir Geliştirme**: Gençlerin proje fikirlerini somutlaştırmaları için mentorluk ve kaynak sağlar.  
- **Eğitim & Atölyeler**: Girişimcilik, teknoloji, tasarım gibi alanlarda pratik eğitimler düzenler (örneğin; yazılım, kodlama, prototip geliştirme).  
- **Ağ Oluşturma**: Katılımcıları yatırımcılar, uzmanlar ve sektör profesyonelleriyle buluşturarak iş bağlantıları kurmalarını destekler.  
- **Fiziksel Altyapı**: Atölyelerde 3D yazıcı, montaj ekipmanları gibi prototip üretim imkânları sunar.  

**İpek Yolu Girişimci Kuluçka Merkezi özelinde**:  
> "Gen

In [ ]:
!zip -r /content/chroma_db.zip /content/chroma_db

from google.colab import files
files.download("/content/chroma_db.zip")

	zip warning: name not matched: /content/chroma_db

zip error: Nothing to do! (try: zip -r /content/chroma_db.zip . -i /content/chroma_db)


FileNotFoundError: Cannot find file: /content/chroma_db.zip

In [ ]:
# Load from BOTH directories
old_data_path = './ipekgpt/data'  # Your old format JSON files
new_data_path = './ipekgpt/IPEKYOLU_RAG_VERISETI'  # Your new format JSON files

chatbot = build_turkish_rag_system_dual_format(
    old_data_path=old_data_path,
    new_data_path=new_data_path
)

print("\n" + "=" * 70)
print("📖 İPEKYOLU RAG MODÜLÜ YÜKLENDİ - Kullanıma hazır!")
print("=" * 70)

In [ ]:
questions = ["İpek Yolu hakkında temel bilgi verebilir misin?"]
batch_test_questions(chatbot, questions)

### Downloading a Folder from Google Colab

To download a folder, we'll first compress it into a `.zip` file and then use Colab's built-in file download utility.

**Instructions:**
1.  Replace `'your_folder_name'` with the actual path to the folder you want to download.
2.  Run the following code cells.

In [ ]:
# Replace 'your_folder_name' with the actual name of the folder you want to download
folder_to_download = 'chroma_db'
output_zip_name = f'{folder_to_download}.zip'

# Zip the folder
!zip -r {output_zip_name} {folder_to_download}

print(f'Folder "{folder_to_download}" has been zipped into "{output_zip_name}"')


In [ ]:
from google.colab import files

# Replace 'your_folder_name' with the actual name of the folder you zipped
folder_to_download = 'chroma_db'
output_zip_name = f'{folder_to_download}.zip'

# Download the zipped folder
try:
    files.download(output_zip_name)
    print(f'Downloading "{output_zip_name}"... Check your browser downloads.')
except Exception as e:
    print(f'Error downloading file: {e}')
    print(f'Please ensure "{output_zip_name}" exists and is accessible.')


In [ ]:
# Optional: Remove the zip file after download
import os

folder_to_download = 'your_folder_name'
output_zip_name = f'{folder_to_download}.zip'

if os.path.exists(output_zip_name):
    os.remove(output_zip_name)
    print(f'Cleaned up: Removed temporary zip file "{output_zip_name}"')
